## Mateus Ribeiro
### Códigos para o trabalho de Cadeias de Markov

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

### Cálculo das matrizes de transições com base na equação de chapman-kolmogorov

In [ ]:
# Definindo a matriz de transição de 1 passo
P = np.array([[0.3, 0.2, 0.25, 0.25],
              [0.3, 0.4, 0.15, 0.15],
              [0.1, 0.2, 0.5, 0.2], 
              [0.1, 0.1, 0.1, 0.7]])

# Definindo uma lista para os valores desejados de "n"
ns = [1, 2, 5, 10, 20, 50, 100]

Ps = [np.linalg.matrix_power(P, n) for n in ns]

In [ ]:
for key, item in zip(ns, Ps):
    print(f"Matriz de transição de {key} passos: \n {item}\n")

### Definindo uma função que calcula a diferença entre as matrizes dos $n's$ consecutivos e a soma do quadrado da diferença entre cada elemento da matriz de diferenças para se ter uma noção "unitária" de convergência

#### Ex:

Diff = $P(100) - P(50)$

udiff = $\sum_i \sum_j (p_{ij} (100) - p_{ij} (50))^2$

In [ ]:
from typing import List, Tuple

def calcular_diffs(Ps: List[np.ndarray]) -> Tuple[List[np.ndarray], List[float]]:
    # Lista para armazenar as matrizes de diferenca
    diffs = []
    for i in range(1, len(Ps)):
        diff = Ps[i] - Ps[i-1]
        diffs.append(diff)

    # Lista para armazenar a soma do quadrado dos elementos 
    # das matrizes de diferenca
    diffs_unitaria = [np.sum(mat**2) for mat in diffs]

    return diffs, diffs_unitaria

In [ ]:
diffs, udiffs = calcular_diffs(Ps)

#### Printando a diferença entre as matrizes para ver a convergência

In [ ]:
for prev, curr, diff in zip(ns[:-1], ns[1:], diffs):
    print(f'Matriz de diferença entre a matriz de {prev} passos e a de {curr} passos: \n {diff}\n')

#### Pritando a diferença "unitária" entre as matrizes 

In [ ]:
for prev, curr, diff in zip(ns[:-1], ns[1:], udiffs):
    print(f'diferença "unitaria" entre a matriz de {prev} passos e a de {curr} passos: {diff}\n')

#### Plotando dos valores das $udiffs$ para visualizar a convergência

In [ ]:
# Definindo os xs para utilizar no plot
xs = np.arange(1, len(udiffs) + 1)
plt.figure(figsize=(10, 6)) 

plt.plot(xs, udiffs, 
         marker='o',          
         markersize=5,        
         linestyle='-',       
         linewidth=1,         
         color="#152361",     
         alpha=1.0)          

plt.title(r'Convergência: Soma dos Quadrados das Diferenças ($P_i - P_{i-1}$)', 
          color="#000000")

plt.grid()
plt.xlabel('Iteração ($i$)')
plt.ylabel('Soma dos Quadrados')

### Calculando a probabilidade de estar em cada estado apos $n$ passos começando de cada estado inicial arbitrário

In [ ]:
estados_iniciais = {
    'Estado E': np.array([1.0, 0.0, 0.0, 0.0]),
    'Estado R': np.array([0.0, 1.0, 0.0, 0.0]),
    'Estado L': np.array([0.0, 0.0, 1.0, 0.0]),
    'Estado A': np.array([0.0, 0.0, 0.0, 1.0])
}

for estado, p0 in estados_iniciais.items():
    print(f"Começando 100% no {estado}")
    
    print(f"n = {0:<3} | p(0) = [{p0[0]:.4f}, {p0[1]:.4f}, {p0[2]:.4f}, {p0[3]:.4f}]")
    for n in ns:
        # Eleva a matriz P a potência n e multiplica pelo vetor inicial
        Pn = np.linalg.matrix_power(P, n)
        pn = np.dot(p0, Pn)
        
        print(f"n = {n:<3} | p(n) = [{pn[0]:.4f}, {pn[1]:.4f}, {pn[2]:.4f}, {pn[3]:.4f}]")
    print("-" * 45)

### Visualizando o vetor de probabilidades estacionárias

In [ ]:
estados = ['E', 'R', 'L', 'A']
probabilidades = [0.1751, 0.2003, 0.2271, 0.3975]
plt.figure(figsize=(8, 5))


plt.bar(estados, probabilidades)


plt.title('Distribuição de Probabilidades Estacionárias')
plt.xlabel('Estados da Cadeia')
plt.ylabel('Probabilidades')

plt.tight_layout()
plt.show()

### Para avaliar o quão rápido o sistema "esquece" as probabilidades iniciais e converge para as estacionárias, iremos utilizar como base a seguinte relação

$p(n) = p(0) P(n) = p(0) P^n$

In [ ]:
# Definindo o vetor de probabilidades iniciais descritos na questao
p0 = np.array([0.3, 0.2, 0.1, 0.4])
powers = np.arange(1, 50, 1)

# Definindo as matrizes de transicao de 1 a 100 passos
Ps = [np.linalg.matrix_power(P, power) for power in powers]
ps = [np.dot(p0, Pi) for Pi in Ps]

In [ ]:
# Pegando as probabilidades individuais dos estados
Es = [p[0] for p in ps]
Rs = [p[1] for p in ps]
Ls = [p[2] for p in ps]
As = [p[3] for p in ps]

In [ ]:
# Pegando o valor final de cada uma
Ey = Es[-1]
Ry = Rs[-1]
Ly = Ls[-1]
Ay = As[-1]
print(Ey, Ry, Ly, Ay)

#### Função para achar o "settling" para cada probabilidade, ou seja, a partir de que iteração a probabilidade está a uma margem de 10% da estacionária

In [ ]:
def find_settling(probs: List, indexes: List ,threshold: float = 0.01):
    final_value = probs[-1]
    upper = (1 + threshold) * final_value
    lower = (1 - threshold) * final_value

    for i in range(len(probs) - 1, -1 ,-1):
        if probs[i] < lower or probs[i] > upper:
            return indexes[i+1]

    return indexes[0]

In [ ]:
settling_E = find_settling(Es, powers)
settling_R = find_settling(Rs, powers)
settling_L = find_settling(Ls, powers)
settling_A = find_settling(As, powers)

print(f"O estado E acomoda no passo: {settling_E}")
print(f"O estado R acomoda no passo: {settling_R}")
print(f"O estado L acomoda no passo: {settling_L}")
print(f"O estado A acomoda no passo: {settling_A}")

### Plotando os valores das probabilidades ao longo dos passos para observar a convergência para os valores estacionários

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Paleta de cores sóbria
cores = {'E': "#117de9", 'R': '#e74c3c', 'L': '#27ae60', 'A': "#dd12f3"}

# Plotando as linhas principais
ax.plot(powers[:10], Es[:10], label='Estado E', color=cores['E'], linewidth=1)
ax.plot(powers[:10], Rs[:10], label='Estado R', color=cores['R'], linewidth=1)
ax.plot(powers[:10], Ls[:10], label='Estado L', color=cores['L'], linewidth=1)
ax.plot(powers[:10], As[:10], label='Estado A', color=cores['A'], linewidth=1)

# Adicionando as assíntotas 
ax.axhline(y=Ey, color=cores['E'], linestyle='--', linewidth=1.5, label = f"Probabilidade estacionaria E: {Ey:.4f}")
ax.axhline(y=Ry, color=cores['R'], linestyle='--', linewidth=1.5, label = f"Probabilidade estacionaria R: {Ry:.4f}")
ax.axhline(y=Ly, color=cores['L'], linestyle='--', linewidth=1.5, label = f"Probabilidade estacionaria L: {Ly:.4f}")
ax.axhline(y=Ay, color=cores['A'], linestyle='--', linewidth=1.5, label = f"Probabilidade estacionaria A: {Ay:.4f}")


ax.axvline(x=settling_E, color=cores['E'], linestyle=':', linewidth=4, alpha=0.5, label = f"Acomodação E (n={settling_E})")  
ax.axvline(x=settling_R, color=cores['R'], linestyle=':', label = f"Acomodação R (n={settling_R})") 
ax.axvline(x=settling_L, color=cores['L'], linestyle=':', label = f"Acomodação L (n={settling_L})")  
ax.axvline(x=settling_A, color=cores['A'], linestyle=':', alpha=1.0, label = f"Acomodação A (n={settling_A})")   

# Título e Eixos formatados
plt.title(r'Convergência do vetor de probabilidade ao longo dos passos', color="#000000")

plt.xlabel("Iteração", fontsize=12, labelpad=10)
plt.ylabel("Probabilidade", fontsize=12, labelpad=10)

# Define os limites dos eixos
plt.xlim(0.5, len(powers[:10]))
plt.ylim(0, 0.5) # Ajuste para 0.5 para focar melhor nos dados

# Legenda fora do gráfico para não sobrepor as linhas
plt.tight_layout()
plt.grid(True)
plt.legend(loc='center left', bbox_to_anchor=(1, 0.5), frameon=False)

plt.show()